# Experimentos — MLP (embeddings, PyTorch) vs. baselines

Notebook de experimentação do MLP (`MLPRecommender`, `src/models/mlp.py`) — fica fora de
`02_experiments.ipynb` por decisão do
[ADR 0003](../docs/experimentos/0003-mlp-fora-do-notebook-02.md): o MLP tem um ciclo de
desenvolvimento próprio (arquitetura, embeddings, curva de treino por época) que não
compartilha o formato "grid de hiperparâmetros + comparação final" usado para os
baselines.

Card Notion `TCF1-157`. Ver `docs/experimentos/0007-0010` para o histórico de decisões
que embasam o design do MLP (paradigma implícito, arquitetura de torre única, early
stopping interno).

In [14]:
try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # O runtime remoto do Colab é uma VM isolada — não enxerga src/ do projeto local nem
    # com sys.path/cwd ajustados (célula seguinte). Precisa clonar o repo pra dentro da VM.
    REPO_URL = "https://github.com/fiap-postech-ml-engineering/ecommerce-recsys-mlops.git"
    REPO_DIR = "/content/ecommerce-recsys-mlops"

    import os

    if not os.path.isdir(REPO_DIR):
        print(f"Clonando {REPO_URL} em {REPO_DIR}...")
        !git clone -q {REPO_URL} {REPO_DIR}
    else:
        print(f"{REPO_DIR} já existe, pulando clone.")

    # uv pip install -e exigiria Python 3.13 (requires-python do pyproject.toml), mas o
    # Colab roda 3.12 — instala os pacotes soltos em vez do projeto editável, ignorando
    # essa checagem de versão.
    print("Instalando dependências (pip, sem checar requires-python do projeto)...")
    !pip install -q torch mlflow matplotlib pandas numpy scikit-learn scikit-surprise implicit pandera kagglehub pydantic-settings python-dotenv

    os.chdir(REPO_DIR)
    print(f"CWD ajustado para {REPO_DIR}")

/content/ecommerce-recsys-mlops já existe, pulando clone.
Instalando dependências (pip, sem checar requires-python do projeto)...
CWD ajustado para /content/ecommerce-recsys-mlops


In [15]:
if IN_COLAB:
    # .env é ignorado no git (tem segredos, .gitignore) — não vem no clone. Preenchido
    # manualmente aqui via getpass (não aparece em texto plano nem fica salvo na saída
    # da célula). Ver .env.example no repo para a lista completa de variáveis.
    from getpass import getpass

    databricks_host = getpass("DATABRICKS_HOST (ex: https://<workspace>.cloud.databricks.com): ")
    databricks_token = getpass("DATABRICKS_TOKEN: ")

    with open(".env", "w") as f:
        f.write(f"DATABRICKS_HOST={databricks_host}\n")
        f.write(f"DATABRICKS_TOKEN={databricks_token}\n")
        f.write("MLFLOW_TRACKING_URI=databricks\n")

    print(".env criado em", os.path.abspath(".env"))

    # data/processed/dataset_consolidated.csv não vem no clone (ignorado no git, versionado
    # via DVC) — sem problema, load_or_build_dataset() (seção 3) já baixa o dataset bruto do
    # Kaggle via kagglehub e reconstrói o CSV processado automaticamente se ele não existir.
    # kagglehub pode pedir autenticação Kaggle (usuário + API key) na primeira chamada.

.env criado em /content/ecommerce-recsys-mlops/.env


In [16]:
import os
from pathlib import Path
import sys

# Jupyter roda com CWD = diretório do notebook, não a raiz do projeto — isso quebra tanto
# o import de src/ quanto a leitura do .env (get_settings() usa caminho relativo). Ajusta
# os dois pra funcionar igual não importa de onde o notebook seja aberto.
ROOT = Path().resolve().parent if Path().resolve().name == "notebooks" else Path().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

## 1. Setup

Imports, `configure_mlflow_tracking()` e `get_settings()`. A hierarquia de 3 níveis de
`start_notebook_run()` (`model_type` → `model_type_test_name` → run com timestamp) já
está documentada em `notebooks/02_experiments.ipynb`, seção 1 — não repetida aqui.

Experimento dedicado (não o de produção): `.../02 - ECOMM_RECSYS - notebook_mlp_training`
— nome fixo, ver `docs/experimentos/0002` e
`docs/internal/mlflow/02_mlflow_boas_praticas.md`.

In [19]:
import torch

from src.config import get_settings
from src.tracking.mlflow_utils import configure_mlflow_tracking

cfg = get_settings()

# Notebook de exploração do MLP — usa o experimento dedicado, não o de produção
# (default de Settings.MLFLOW_EXPERIMENT_NAME). Ver
# docs/internal/mlflow/02_mlflow_boas_praticas.md, seção 1.
configure_mlflow_tracking(
    experiment_name="/Shared/mlflow_ecomm_recsys/02 - ECOMM_RECSYS - notebook_mlp_training"
)

# MLPRecommender detecta CUDA automaticamente e cai para CPU se indisponível
# (src/models/mlp.py) — só avisa aqui qual device foi escolhido para este ambiente.
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

If you are using MLflow Tracing, you can migrate your traces to Unity Catalog for unlimited storage, fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/migrate-traces-to-uc
2026-07-20 17:11:43,206 - [INFO    ] - src.tracking.mlflow_utils - mlflow_utils:84        - request_id=-                                    - MLflow Databricks conectado
2026-07-20 17:11:43,208 - [INFO    ] - src.tracking.mlflow_utils - mlflow_utils:85        - request_id=-                                    - Experimento: /Shared/mlflow_ecomm_recsys/02 - ECOMM_RECSYS - notebook_mlp_training


Device: cuda


## 2. Recuperar o vencedor dos baselines (notebook 02)

`notebooks/02_experiments.ipynb` já decide, por execução programática (nunca hardcoded),
qual modelo puro (SVD/ALS/BPR/ItemKNN) vence por NDCG@10 (critério principal) / Hit
Rate@10 (desempate) — ver seção 9 daquele notebook. Esta seção consulta o MLflow para
recuperar esse vencedor e a configuração tunada (fase `final`), em vez de hardcodar um
nome — mesma disciplina de não assumir o resultado de antemão.

**Pré-requisito**: `02_experiments.ipynb` precisa ter rodado até o fim pelo menos uma vez
(runs `phase="final"` já logadas no experimento `notebook_baselines_training`). Se não
tiver rodado, a célula abaixo levanta um erro claro.

In [23]:
import mlflow
from mlflow.tracking import MlflowClient

BASELINES_EXPERIMENT = (
    "/Shared/mlflow_ecomm_recsys/02 - ECOMM_RECSYS - notebook_baselines_training"
)


def _coerce_param(value: str):
    """Converte um valor de param do MLflow (sempre string) de volta pra int/float/str."""
    for cast in (int, float):
        try:
            return cast(value)
        except ValueError:
            continue
    return value


client = MlflowClient()
baselines_experiment = client.get_experiment_by_name(BASELINES_EXPERIMENT)
if baselines_experiment is None:
    raise RuntimeError(
        f"Experimento '{BASELINES_EXPERIMENT}' não encontrado. "
        "Rode notebooks/02_experiments.ipynb até o fim antes deste notebook."
    )

final_baseline_runs = client.search_runs(
    experiment_ids=[baselines_experiment.experiment_id],
)
if not final_baseline_runs:
    raise RuntimeError(
        "Nenhuma run 'final' encontrada em notebook_baselines_training. "
        "Rode notebooks/02_experiments.ipynb até o fim antes deste notebook."
    )

# search_runs sem filtro traz também as pastas de agrupamento vazias que
# start_notebook_run() cria (model_type/model_type_test_name, sempre "Running", sem tags
# — ver notebooks/02_experiments.ipynb, seção 1) — ignoradas aqui via .get() em vez de
# indexação direta, que quebraria com KeyError nelas.
baseline_runs_by_model = {
    run.data.tags["model_type"]: run
    for run in final_baseline_runs
    if run.data.tags.get("model_type") is not None
}
baseline_runs_by_model.pop("popularity", None)

best_baseline_model_type = max(
    baseline_runs_by_model,
    key=lambda name: (
        baseline_runs_by_model[name].data.metrics["model.ndcg_at_k"],
        baseline_runs_by_model[name].data.metrics["model.hit_rate_at_k"],
    ),
)
best_baseline_run = baseline_runs_by_model[best_baseline_model_type]
best_baseline_config = {
    key: _coerce_param(value)
    for key, value in best_baseline_run.data.params.items()
    if key != "recommendation_k"
}

print(f"Vencedor recuperado do notebook 02: {best_baseline_model_type} = {best_baseline_config}")

Vencedor recuperado do notebook 02: svd = {'model': 'svd', 'n_epochs': 20, 'n_factors': 25}


## 3. Carregamento e preparação dos dados

Mesmo pipeline de `02_experiments.ipynb`, seção 2:
`load_or_build_dataset → build_interactions → temporal_split → k_core_filter`.

O MLP treina em rótulo binário implícito (presença de interação), não usa a magnitude do
`score` — `train_df_implicit` (log1p, ver `docs/experimentos/0005`) só é necessário para
refazer o fit do vencedor recuperado na seção 2 (SVD/ALS/BPR/ItemKNN), não para o MLP.

In [24]:
from src.data.filtering import k_core_filter
from src.data.loader import load_or_build_dataset
from src.data.preprocessor import apply_log_scaling, build_interactions
from src.data.split import temporal_split

events = load_or_build_dataset(force_rebuild=False)
interactions = build_interactions(events)

train_df, val_df, test_df = temporal_split(
    interactions,
    test_size=cfg.TEST_SIZE,
    validation_size=cfg.VALIDATION_SIZE,
)

# k-core filtering: remove usuários/itens com poucas interações no treino
# (notebooks/01_eda.ipynb documenta a sparsity que motiva o filtro). Threshold
# calculado só a partir do train_df, sem olhar val_df/test_df.
train_df, val_df, test_df = k_core_filter(
    train_df,
    val_df,
    test_df,
    min_user_interactions=cfg.MIN_USER_INTERACTIONS,
    min_item_interactions=cfg.MIN_ITEM_INTERACTIONS,
)

# Só usado pra refazer o fit do vencedor dos baselines (seção 2) — o MLP (src/models/mlp.py)
# não lê a coluna score, treina em presença/ausência de interação (ver docs/experimentos/0010).
train_df_implicit = apply_log_scaling(train_df)

catalog_size = interactions["item_id"].nunique()

print(
    f"Interações:   {len(interactions):>9,}  ({catalog_size:,} itens, {interactions['user_id'].nunique():,} usuários)"
)
print(f"Treino:       {len(train_df):>9,}  até {train_df['timestamp'].max()}")
print(
    f"Validação:    {len(val_df):>9,}  {val_df['timestamp'].min()} → {val_df['timestamp'].max()}"
)
print(f"Teste:        {len(test_df):>9,}  a partir de {test_df['timestamp'].min()}")

2026-07-20 17:13:03,797 - [INFO    ] - src.data.loader           - loader:42              - request_id=-                                    - Baixando dataset RetailRocket via kagglehub (faltam: ['events.csv', 'category_tree.csv', 'item_properties_part1.csv', 'item_properties_part2.csv'])


Using Colab cache for faster access to the 'ecommerce-dataset' dataset.


2026-07-20 17:14:10,360 - [INFO    ] - src.data.loader           - loader:99              - request_id=-                                    - Dataset consolidado salvo em: /content/ecommerce-recsys-mlops/data/processed/dataset_consolidated.csv
2026-07-20 17:14:12,885 - [INFO    ] - src.data.split            - split:36               - request_id=-                                    - Split temporal: 1151589 treino, 383863 validação, 383864 teste, 253353 usuários cold-start no teste
2026-07-20 17:14:13,799 - [INFO    ] - src.data.filtering        - filtering:73           - request_id=-                                    - k-core filtering: treino 1151589 -> 399022 linhas, 110199 usuários, 24016 itens


Interações:   1,919,316  (183,681 itens, 1,232,387 usuários)
Treino:         399,022  até 2015-07-22 17:14:22.187000
Validação:       12,702  2015-07-22 17:14:43.373000 → 2015-08-18 18:41:55.701000
Teste:            7,817  a partir de 2015-08-18 18:44:59.447000


## 4. Loop de avaliação

Cópia local de `evaluate_model` (`notebooks/02_experiments.ipynb`, seção 3) — sem
`src/evaluation/` compartilhado ainda (criar esse módulo está fora do escopo deste
notebook), mas mesma função, mesma assinatura, mesmo ground truth (`transaction`).

**Métricas retornadas:** Precision@K, Recall@K, NDCG@K, Hit Rate@K, Coverage, Revenue@K

In [25]:
import statistics

from src.data.preprocessor import EVENT_WEIGHTS
from src.metrics.business import coverage, revenue_at_k
from src.metrics.ranking import hit_rate_at_k, ndcg_at_k, precision_at_k, recall_at_k

K = cfg.RECOMMENDATION_K

# build_interactions() perde o tipo de evento na agregação (score = soma de pesos por par
# user/item). score >= peso de "transaction" aproxima "houve transação" — mesmo proxy do
# notebook 02, derivado de EVENT_WEIGHTS em vez de fixo, pra acompanhar mudanças ali.
TRANSACTION_SCORE_THRESHOLD = EVENT_WEIGHTS["transaction"]


def evaluate_model(model, split_df, train_df, k):
    """Avalia um BaseRecommender já treinado sobre split_df, retornando as 6 métricas."""
    eval_users = set(split_df["user_id"]) & set(train_df["user_id"])
    purchases = split_df[split_df["score"] >= TRANSACTION_SCORE_THRESHOLD]
    relevant_by_user = purchases.groupby("user_id")["item_id"].apply(set).to_dict()
    value_by_item = split_df.drop_duplicates("item_id").set_index("item_id")["value"].to_dict()

    scores = {"precision": [], "recall": [], "ndcg": [], "hit_rate": [], "revenue": []}
    all_recs = []

    for user_id in eval_users:
        relevant = relevant_by_user.get(user_id, set())
        if not relevant:
            continue
        recs = model.recommend(user_id=user_id, k=k)
        all_recs.append(recs)
        relevant_with_value = {i: value_by_item[i] for i in relevant if i in value_by_item}

        scores["precision"].append(precision_at_k(recs, relevant, k))
        scores["recall"].append(recall_at_k(recs, relevant, k))
        scores["ndcg"].append(ndcg_at_k(recs, relevant, k))
        scores["hit_rate"].append(hit_rate_at_k(recs, relevant, k))
        scores["revenue"].append(revenue_at_k(recs, relevant_with_value, k))

    return {
        "precision": statistics.mean(scores["precision"]),
        "recall": statistics.mean(scores["recall"]),
        "ndcg": statistics.mean(scores["ndcg"]),
        "hit_rate": statistics.mean(scores["hit_rate"]),
        "coverage": coverage(all_recs, catalog_size),
        "revenue": sum(scores["revenue"]),
    }

## 5. Exploração do MLP (dev)

Poucas variações de hiperparâmetros (arquitetura, negative sampling, learning rate) como
runs `phase="dev"`, avaliadas no `val_df` — objetivo é "definir hiperparâmetros que
convergem" (card TCF1-157), não uma busca exaustiva. Cada run também loga a curva de loss
(`training_history`, train vs. val) como artefato de diagnóstico
(`diagnostics/training_curve.png`).

In [26]:
from pathlib import Path
import tempfile

import matplotlib.pyplot as plt

from src.models.factory import RecommenderFactory
from src.tracking.mlflow_utils import log_evaluation_metrics, start_notebook_run

DEV_CONFIGS = {
    "hd128-64_ns4": {"hidden_dims": [128, 64], "negative_samples": 4},
    "hd128-64_ns8": {"hidden_dims": [128, 64], "negative_samples": 8},
    "hd256-128-64_ns4": {"hidden_dims": [256, 128, 64], "negative_samples": 4},
    "hd64_ns4_lr01": {"hidden_dims": [64], "negative_samples": 4, "learning_rate": 0.01},
}


def _log_training_curve(model) -> None:
    """Plota train/val loss por época e loga como artefato de diagnóstico no run ativo."""
    fig, ax = plt.subplots()
    ax.plot(model.training_history["train_loss"], label="train_loss")
    ax.plot(model.training_history["val_loss"], label="val_loss")
    ax.set_xlabel("época")
    ax.set_ylabel("BCE loss")
    ax.legend()
    with tempfile.TemporaryDirectory() as tmp_dir:
        path = Path(tmp_dir) / "training_curve.png"
        fig.savefig(path)
        mlflow.log_artifact(str(path), artifact_path="diagnostics")
    plt.close(fig)


dev_results = {}
total_configs = len(DEV_CONFIGS)
for i, (test_name, config) in enumerate(DEV_CONFIGS.items(), start=1):
    print(f"[{i}/{total_configs}] treinando mlp({test_name}) — config={config}...")

    model = RecommenderFactory.create("mlp", config)
    model.fit(train_df)
    print(
        f"[{i}/{total_configs}] {test_name}: fit concluído, epochs_trained={model.epochs_trained}/{model.epochs}"
    )

    with start_notebook_run(
        model_type="mlp",
        test_name=test_name,
        phase="dev",
        dataset_name="retailrocket",
        params=model.get_params(),
    ):
        mlflow.set_tag(
            "mlflow.note.content",
            f"MLP dev — hidden_dims={model.hidden_dims}, negative_samples={model.negative_samples}, "
            f"epochs_trained={model.epochs_trained}/{model.epochs}.",
        )
        metrics = evaluate_model(model, val_df, train_df, K)
        log_evaluation_metrics(metrics, K)
        _log_training_curve(model)

    dev_results[test_name] = {"config": config, "metrics": metrics}
    print(
        f"[{i}/{total_configs}] {test_name}: NDCG@{K}={metrics['ndcg']:.4f}  HitRate@{K}={metrics['hit_rate']:.4f}  "
        f"epochs_trained={model.epochs_trained}\n"
    )

best_dev_name = max(
    dev_results,
    key=lambda name: (
        dev_results[name]["metrics"]["ndcg"],
        dev_results[name]["metrics"]["hit_rate"],
    ),
)
best_dev_config = dev_results[best_dev_name]["config"]
print(f"Melhor configuração de dev: {best_dev_name} = {best_dev_config}")

[1/4] treinando mlp(hd128-64_ns4) — config={'hidden_dims': [128, 64], 'negative_samples': 4}...


/usr/local/lib/python3.12/dist-packages/implicit/gpu/__init__.py:28: UserWarning: Disabling GPU support because of 'libcublas.so.13: cannot open shared object file: No such file or directory'
  warnings.warn(


NotImplementedError: 

## 6. Configuração escolhida (tuning)

Um run adicional com a melhor configuração de dev, agora `phase="tuning"` — mesmo
raciocínio de "promover só o vencedor" usado no grid dos baselines (`02_experiments.ipynb`,
seção 10). Ainda avaliado no `val_df` (o `test_df` só é tocado na comparação final,
seção 7).

In [ ]:
mlp_model = RecommenderFactory.create("mlp", best_dev_config)
mlp_model.fit(train_df)

with start_notebook_run(
    model_type="mlp",
    test_name=f"final_{best_dev_name}",
    phase="tuning",
    dataset_name="retailrocket",
    params=mlp_model.get_params(),
):
    mlp_tuning_metrics = evaluate_model(mlp_model, val_df, train_df, K)
    log_evaluation_metrics(mlp_tuning_metrics, K)
    _log_training_curve(mlp_model)

mlp_tuning_metrics

## 7. Comparativo final

Popularity (piso) + vencedor recuperado dos baselines (seção 2, refeito em
`train_df_implicit`) + MLP tunado (seção 6, refeito em `train_df`), todos avaliados uma
única vez no `test_df`, `phase="final"`. Este resultado responde ao card TCF1-157 e
alimenta `src/training/train.py`.

In [ ]:
import pandas as pd

FINAL_MODELS = {
    "popularity": ("default", {}, train_df),
    best_baseline_model_type: (
        f"final_{best_baseline_model_type}",
        best_baseline_config,
        train_df_implicit,
    ),
    "mlp": (f"final_{best_dev_name}", best_dev_config, train_df),
}

for model_type, (test_name, model_config, model_train_df) in FINAL_MODELS.items():
    model = RecommenderFactory.create(model_type, model_config)
    model.fit(model_train_df)

    with start_notebook_run(
        model_type=model_type,
        test_name=test_name,
        phase="final",
        dataset_name="retailrocket",
        params=model.get_params(),
    ):
        final_metrics = evaluate_model(model, test_df, train_df, K)
        log_evaluation_metrics(final_metrics, K)
        if hasattr(model, "training_history"):
            _log_training_curve(model)

client = MlflowClient()
mlp_experiment_id = mlflow.tracking.fluent._get_experiment_id()
final_runs = client.search_runs(
    experiment_ids=[mlp_experiment_id], filter_string="tags.phase = 'final'"
)

comparison_df = pd.DataFrame(
    {run.data.tags["model_type"]: run.data.metrics for run in final_runs}
).T
comparison_df

## 8. Conclusão

A tabela acima compara Popularity (piso de personalização), o melhor baseline clássico
recuperado do notebook 02 (SVD/ALS/BPR/ItemKNN, já tunado) e o MLP (embeddings + torre
MLP, tunado nas seções 5-6) nas mesmas 6 métricas, sobre o mesmo `test_df`. Esse
comparativo — não um único número isolado — é o critério de aceite do card `TCF1-157` e o
que `src/training/train.py` vai formalizar como pipeline de treino oficial.